In [17]:
# Import libraries
import pandas as pd
import numpy as np
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

## 1. Custom Transformers

Custom sklearn-compatible transformers that fit on training data and apply consistently.

In [18]:
class GroupImputer(BaseEstimator, TransformerMixin):
    """
    Sklearn-compatible imputer that uses group-based statistics.
    Fits on training data, applies consistently to any new data.
    """
    
    def __init__(self, group_col='Industry'):
        self.group_col = group_col
        self.group_stats_ = {}
        self.global_stats_ = {}
        self.numeric_cols_ = []
        self.categorical_cols_ = []
    
    def fit(self, X, y=None):
        """Fit imputer by calculating group statistics from training data."""
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Identify column types
        self.numeric_cols_ = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
        self.categorical_cols_ = df.select_dtypes(include=['object']).columns.tolist()
        
        # Remove group column from imputation targets
        if self.group_col in self.numeric_cols_:
            self.numeric_cols_.remove(self.group_col)
        if self.group_col in self.categorical_cols_:
            self.categorical_cols_.remove(self.group_col)
        
        # Calculate group statistics for numeric columns (median)
        for col in self.numeric_cols_:
            self.group_stats_[col] = df.groupby(self.group_col)[col].median().to_dict()
            self.global_stats_[col] = df[col].median()
        
        # Calculate group statistics for categorical columns (mode)
        for col in self.categorical_cols_:
            group_modes = df.groupby(self.group_col)[col].agg(
                lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan
            ).to_dict()
            self.group_stats_[col] = group_modes
            mode_result = df[col].mode()
            self.global_stats_[col] = mode_result.iloc[0] if len(mode_result) > 0 else np.nan
        
        return self
    
    def transform(self, X):
        """Apply imputation using fitted statistics."""
        df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        for col in self.numeric_cols_ + self.categorical_cols_:
            if col not in df.columns:
                continue
            
            group_values = self.group_stats_.get(col, {})
            global_value = self.global_stats_.get(col)
            
            # Apply group-based imputation
            for idx in df[df[col].isna()].index:
                group = df.loc[idx, self.group_col]
                if pd.notna(group) and group in group_values and pd.notna(group_values[group]):
                    df.loc[idx, col] = group_values[group]
                else:
                    df.loc[idx, col] = global_value
        
        return df

In [19]:
class CategoryGrouper(BaseEstimator, TransformerMixin):
    """
    Groups infrequent categories into 'Other' based on training data frequency.
    Fits threshold on training data, applies consistently to new data.
    """
    
    def __init__(self, column='Country', threshold=0.8, other_label='Other'):
        self.column = column
        self.threshold = threshold
        self.other_label = other_label
        self.top_categories_ = None
    
    def fit(self, X, y=None):
        """Fit by determining top categories from training data."""
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        if self.column not in df.columns:
            self.top_categories_ = set()
            return self
        
        value_counts = df[self.column].value_counts()
        cumulative_pct = value_counts.cumsum() / len(df)
        
        # Find categories up to threshold
        self.top_categories_ = set(cumulative_pct[cumulative_pct <= self.threshold].index)
        
        # Include the category that crosses the threshold
        if len(self.top_categories_) < len(value_counts):
            remaining = cumulative_pct[cumulative_pct > self.threshold]
            if len(remaining) > 0:
                self.top_categories_.add(remaining.index[0])
        
        return self
    
    def transform(self, X):
        """Apply category grouping."""
        df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        if self.column not in df.columns:
            return df
        
        new_col = f"{self.column}_grouped"
        df[new_col] = df[self.column].apply(
            lambda x: x if x in self.top_categories_ else self.other_label
        )
        
        # Drop original column
        df = df.drop(columns=[self.column])
        
        return df

In [20]:
class ColumnDropper(BaseEstimator, TransformerMixin):
    """Drops specified columns. Useful for removing leakage features."""
    
    def __init__(self, columns=None):
        self.columns = columns or []
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        cols_to_drop = [c for c in self.columns if c in df.columns]
        return df.drop(columns=cols_to_drop)

In [21]:
class LogTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p transformation to specified numeric columns."""
    
    def __init__(self, columns=None, drop_original=True):
        self.columns = columns or []
        self.drop_original = drop_original
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        for col in self.columns:
            if col in df.columns:
                new_col = f'log_{col.lower().replace("company_annual_", "")}'
                df[new_col] = np.log1p(df[col].clip(lower=0))
                if self.drop_original:
                    df = df.drop(columns=[col])
        
        return df

In [22]:
class DataFrameEncoder(BaseEstimator, TransformerMixin):
    """
    Wraps ColumnTransformer to maintain feature names and output DataFrame.
    """
    
    def __init__(self, ordinal_features, ordinal_mappings, nominal_features, numeric_features):
        self.ordinal_features = ordinal_features
        self.ordinal_mappings = ordinal_mappings
        self.nominal_features = nominal_features
        self.numeric_features = numeric_features
        self.column_transformer_ = None
        self.feature_names_ = None
    
    def fit(self, X, y=None):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Build ordinal categories in correct order
        ordinal_categories = [self.ordinal_mappings[col] for col in self.ordinal_features 
                             if col in self.ordinal_mappings]
        
        self.column_transformer_ = ColumnTransformer(
            transformers=[
                ('ordinal', OrdinalEncoder(
                    categories=ordinal_categories,
                    handle_unknown='use_encoded_value',
                    unknown_value=-1
                ), self.ordinal_features),
                
                ('nominal', OneHotEncoder(
                    handle_unknown='ignore',
                    sparse_output=False
                ), self.nominal_features),
                
                ('numeric', StandardScaler(), self.numeric_features),
            ],
            remainder='drop'
        )
        
        self.column_transformer_.fit(df)
        
        # Get feature names
        self.feature_names_ = self._get_feature_names()
        
        return self
    
    def transform(self, X):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        return self.column_transformer_.transform(df)
    
    def _get_feature_names(self):
        """Extract feature names from fitted transformer."""
        names = []
        
        # Ordinal features keep their names
        names.extend(self.ordinal_features)
        
        # OneHot features get expanded names
        ohe = self.column_transformer_.named_transformers_['nominal']
        for i, col in enumerate(self.nominal_features):
            for cat in ohe.categories_[i]:
                names.append(f"{col}_{cat}")
        
        # Numeric features keep their names
        names.extend(self.numeric_features)
        
        return names

## 2. Load Raw Data (Post-Split)

Load the preprocessed train/val/test splits that have imputation done correctly.

In [40]:
# Load preprocessed data
train_df = pd.read_csv('../data/preprocessed/train.csv')
val_df = pd.read_csv('../data/preprocessed/val.csv')
test_df = pd.read_csv('../data/preprocessed/test.csv')

TARGET_COL = 'Scope_3_coverage'

X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]

X_val = val_df.drop(columns=[TARGET_COL])
y_val = val_df[TARGET_COL]

X_test = test_df.drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

print(f"Training:   {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")
print(f"\nColumns: {X_train.columns.tolist()}")

Training:   (1131, 22)
Validation: (377, 22)
Test:       (377, 22)

Columns: ['Country', 'Geographic_region', 'Private_company', 'End_target', 'End_target_year', 'Status_of_end_target', 'Interim_target', 'Interim_target_year', 'GHGs_covered', 'Scope_1_coverage', 'Scope_2_coverage', 'Published_plan', 'Reporting_mechanism', 'Accountability_delivery', 'Carbon_credits', 'Separate_removal_target', 'Planning_removals', 'Historical_emissions', 'Race_to_zero_member', 'Industry', 'log_revenue', 'log_employees']


## 3. Define Pipeline Configuration

In [41]:
# ============================================================
# PIPELINE CONFIGURATION
# ============================================================

# Features to drop (data leakage)
LEAKAGE_COLS = ['Scope_1_coverage', 'Scope_2_coverage']

# Country grouping
COUNTRY_THRESHOLD = 0.8

# Ordinal features and their mappings
ORDINAL_FEATURES = [
    'Published_plan',
    'Private_company', 
    'Race_to_zero_member',
    'Separate_removal_target',
    'Historical_emissions',
    'Accountability_delivery',
    'Carbon_credits',
    'GHGs_covered',
    'Reporting_mechanism',
    'Status_of_end_target',
    'Planning_removals',
]

ORDINAL_MAPPINGS = {
    'Published_plan': ['No', 'Yes'],
    'Private_company': ['No', 'Yes'],
    'Race_to_zero_member': ['No', 'Yes'],
    'Separate_removal_target': ['No', 'Yes'],
    'Historical_emissions': ['No', 'Yes'],
    'Accountability_delivery': ['Not Specified', 'No', 'Yes'],
    'Carbon_credits': ['No', 'Not Specified', 'Yes'],
    'GHGs_covered': ['Not Specified', 'Carbon dioxide only', 'Carbon dioxide and other GHGs'],
    'Reporting_mechanism': ['No reporting mechanism', 'Less than annual reporting', 'Annual reporting'],
    'Status_of_end_target': [
        'Proposed / in discussion', 
        'Declaration / pledge', 
        'In corporate strategy', 
        'Achieved (self-declared)', 
        'Achieved (externally validated)'
    ],
    'Planning_removals': [
        'No', 
        'Not Specified', 
        'Yes (unspecified)', 
        'Yes (nature-based removals e.g. Forestation, soil carbon enhancement)',
        'Yes (CCS-based removals e.g. BECCS, DACCS)',
        'Yes (nature-based and CCS-based removals)'
    ],
}

# Nominal features (after country grouping)
NOMINAL_FEATURES = [
    'Country_grouped',
    'Geographic_region',
    'Industry',
    'End_target',
    'Interim_target',
]

# Numeric features
NUMERIC_FEATURES = [
    'End_target_year',
    'Interim_target_year',
    'log_revenue',
    'log_employees',
]

print("Pipeline Configuration:")
print(f"   Leakage columns to drop: {LEAKAGE_COLS}")
print(f"   Country threshold: {COUNTRY_THRESHOLD}")
print(f"   Ordinal features: {len(ORDINAL_FEATURES)}")
print(f"   Nominal features: {len(NOMINAL_FEATURES)}")
print(f"   Numeric features: {len(NUMERIC_FEATURES)}")

Pipeline Configuration:
   Leakage columns to drop: ['Scope_1_coverage', 'Scope_2_coverage']
   Country threshold: 0.8
   Ordinal features: 11
   Nominal features: 5
   Numeric features: 4


## 4. Build Preprocessing Pipeline

In [42]:
# Build the preprocessing pipeline
preprocessing_pipeline = Pipeline([
    ('drop_leakage', ColumnDropper(columns=LEAKAGE_COLS)),
    ('country_grouper', CategoryGrouper(column='Country', threshold=COUNTRY_THRESHOLD)),
])

# Fit on training data
X_train_preprocessed = preprocessing_pipeline.fit_transform(X_train)
X_val_preprocessed = preprocessing_pipeline.transform(X_val)
X_test_preprocessed = preprocessing_pipeline.transform(X_test)

print("✅ Preprocessing Pipeline Fitted")
print(f"   Shape after preprocessing: {X_train_preprocessed.shape}")
print(f"   Columns: {X_train_preprocessed.columns.tolist()}")

✅ Preprocessing Pipeline Fitted
   Shape after preprocessing: (1131, 20)
   Columns: ['Geographic_region', 'Private_company', 'End_target', 'End_target_year', 'Status_of_end_target', 'Interim_target', 'Interim_target_year', 'GHGs_covered', 'Published_plan', 'Reporting_mechanism', 'Accountability_delivery', 'Carbon_credits', 'Separate_removal_target', 'Planning_removals', 'Historical_emissions', 'Race_to_zero_member', 'Industry', 'log_revenue', 'log_employees', 'Country_grouped']


In [43]:
# Build the encoding pipeline
encoder = DataFrameEncoder(
    ordinal_features=ORDINAL_FEATURES,
    ordinal_mappings=ORDINAL_MAPPINGS,
    nominal_features=NOMINAL_FEATURES,
    numeric_features=NUMERIC_FEATURES
)

# Fit on training data
X_train_encoded = encoder.fit_transform(X_train_preprocessed)
X_val_encoded = encoder.transform(X_val_preprocessed)
X_test_encoded = encoder.transform(X_test_preprocessed)

print("✅ Encoder Fitted")
print(f"   Shape after encoding: {X_train_encoded.shape}")
print(f"   Feature names: {len(encoder.feature_names_)} features")

✅ Encoder Fitted
   Shape after encoding: (1131, 72)
   Feature names: 72 features


In [44]:
# Encode target
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

print("Target Classes:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"   {i}: {cls}")

Target Classes:
   0: No
   1: Not Specified
   2: Partial
   3: Yes


## 5. Train Model

In [45]:
# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_encoded, y_train_encoded)
print("✅ Random Forest Trained")
print(f"   Training samples: {len(y_train_encoded)}")

✅ Random Forest Trained
   Training samples: 1131


In [29]:
# Evaluate on validation set
y_val_pred = rf_model.predict(X_val_encoded)

print("📊 Validation Set Performance:")
print("=" * 60)
print(classification_report(y_val_encoded, y_val_pred, target_names=label_encoder.classes_))

📊 Validation Set Performance:
               precision    recall  f1-score   support

           No       0.47      0.39      0.42        70
Not Specified       0.62      0.59      0.60        82
      Partial       0.25      0.10      0.14        73
          Yes       0.55      0.78      0.65       152

     accuracy                           0.53       377
    macro avg       0.47      0.46      0.45       377
 weighted avg       0.49      0.53      0.50       377



In [46]:
# Final evaluation on test set
y_test_pred = rf_model.predict(X_test_encoded)

print("📊 Test Set Performance (Final):")
print("=" * 60)
print(classification_report(y_test_encoded, y_test_pred, target_names=label_encoder.classes_))

📊 Test Set Performance (Final):
               precision    recall  f1-score   support

           No       0.45      0.39      0.42        69
Not Specified       0.63      0.53      0.58        83
      Partial       0.32      0.10      0.15        73
          Yes       0.56      0.83      0.67       152

     accuracy                           0.54       377
    macro avg       0.49      0.46      0.45       377
 weighted avg       0.51      0.54      0.50       377



## 6. Save Pipeline for Production

Save all fitted components so they can be loaded for inference on new data.

In [47]:
# Save all pipeline components
import os

MODEL_DIR = '../models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Save components
joblib.dump(preprocessing_pipeline, f'{MODEL_DIR}/preprocessing_pipeline.pkl')
joblib.dump(encoder, f'{MODEL_DIR}/encoder.pkl')
joblib.dump(label_encoder, f'{MODEL_DIR}/label_encoder.pkl')
joblib.dump(rf_model, f'{MODEL_DIR}/rf_model.pkl')

# Save configuration
config = {
    'LEAKAGE_COLS': LEAKAGE_COLS,
    'COUNTRY_THRESHOLD': COUNTRY_THRESHOLD,
    'ORDINAL_FEATURES': ORDINAL_FEATURES,
    'ORDINAL_MAPPINGS': ORDINAL_MAPPINGS,
    'NOMINAL_FEATURES': NOMINAL_FEATURES,
    'NUMERIC_FEATURES': NUMERIC_FEATURES,
    'TARGET_COL': TARGET_COL,
}
joblib.dump(config, f'{MODEL_DIR}/config.pkl')

print("✅ Pipeline Saved!")
print(f"   Location: {MODEL_DIR}/")
print(f"   Files:")
print(f"      - preprocessing_pipeline.pkl")
print(f"      - encoder.pkl")
print(f"      - label_encoder.pkl")
print(f"      - rf_model.pkl")
print(f"      - config.pkl")

✅ Pipeline Saved!
   Location: ../models/
   Files:
      - preprocessing_pipeline.pkl
      - encoder.pkl
      - label_encoder.pkl
      - rf_model.pkl
      - config.pkl


## 7. Inference Function

Load and use the saved pipeline for predictions on new data.

In [16]:
def load_pipeline(model_dir='../models'):
    """Load saved pipeline components."""
    preprocessing = joblib.load(f'{model_dir}/preprocessing_pipeline.pkl')
    encoder = joblib.load(f'{model_dir}/encoder.pkl')
    label_encoder = joblib.load(f'{model_dir}/label_encoder.pkl')
    model = joblib.load(f'{model_dir}/rf_model.pkl')
    config = joblib.load(f'{model_dir}/config.pkl')
    
    return {
        'preprocessing': preprocessing,
        'encoder': encoder,
        'label_encoder': label_encoder,
        'model': model,
        'config': config
    }

def predict(X_new, pipeline_components):
    """
    Make predictions on new data using the fitted pipeline.
    
    Args:
        X_new: DataFrame with same columns as training data
        pipeline_components: Dict from load_pipeline()
    
    Returns:
        predictions: Array of predicted class labels
        probabilities: Array of class probabilities
    """
    # Preprocess
    X_preprocessed = pipeline_components['preprocessing'].transform(X_new)
    
    # Encode
    X_encoded = pipeline_components['encoder'].transform(X_preprocessed)
    
    # Predict
    y_pred_encoded = pipeline_components['model'].predict(X_encoded)
    y_proba = pipeline_components['model'].predict_proba(X_encoded)
    
    # Decode labels
    predictions = pipeline_components['label_encoder'].inverse_transform(y_pred_encoded)
    
    return predictions, y_proba

# Test inference function
print("🔄 Testing Inference Pipeline...")
pipeline = load_pipeline()

# Use first 5 test samples as "new data"
X_sample = X_test.head(5)
predictions, probabilities = predict(X_sample, pipeline)

print("\nSample Predictions:")
for i, (pred, actual) in enumerate(zip(predictions, y_test.head(5))):
    print(f"   Sample {i+1}: Predicted={pred}, Actual={actual}")

print("\n✅ Inference pipeline working correctly!")

🔄 Testing Inference Pipeline...

Sample Predictions:
   Sample 1: Predicted=Not Specified, Actual=Yes
   Sample 2: Predicted=Yes, Actual=Partial
   Sample 3: Predicted=Yes, Actual=Yes
   Sample 4: Predicted=Yes, Actual=Yes
   Sample 5: Predicted=Yes, Actual=Yes

✅ Inference pipeline working correctly!


## Summary

This notebook implements a **fully reproducible ML pipeline**:

| Component | Purpose | Saved As |
|-----------|---------|----------|
| `ColumnDropper` | Remove leakage features | Part of `preprocessing_pipeline.pkl` |
| `CategoryGrouper` | Group rare countries | Part of `preprocessing_pipeline.pkl` |
| `DataFrameEncoder` | Ordinal + OneHot + Scaling | `encoder.pkl` |
| `LabelEncoder` | Encode target classes | `label_encoder.pkl` |
| `RandomForestClassifier` | Classification model | `rf_model.pkl` |
| Configuration | All hyperparameters | `config.pkl` |

### Usage for New Data:
```python
# Load pipeline
pipeline = load_pipeline('../models')

# Predict on new data
predictions, probabilities = predict(new_data, pipeline)
```

### Key Properties:
1. ✅ **Fit once, apply anywhere** - All transformers fitted on training data
2. ✅ **No data leakage** - Leakage features dropped before any processing
3. ✅ **Reproducible** - Saved and loadable with joblib
4. ✅ **Production-ready** - Single function for inference

## Summary

This notebook implements a **fully reproducible ML pipeline**:

| Component | Purpose | Saved As |
|-----------|---------|----------|
| `ColumnDropper` | Remove leakage features | Part of `preprocessing_pipeline.pkl` |
| `CategoryGrouper` | Group rare countries | Part of `preprocessing_pipeline.pkl` |
| `DataFrameEncoder` | Ordinal + OneHot + Scaling | `encoder.pkl` |
| `LabelEncoder` | Encode target classes | `label_encoder.pkl` |
| `RandomForestClassifier` | Classification model | `rf_model.pkl` |
| Configuration | All hyperparameters | `config.pkl` |

### Usage for New Data:
```python
# Load pipeline
pipeline = load_pipeline('../models')

# Predict on new data
predictions, probabilities = predict(new_data, pipeline)
```

### Key Properties:
1. ✅ **Fit once, apply anywhere** - All transformers fitted on training data
2. ✅ **No data leakage** - Leakage features dropped before any processing
3. ✅ **Reproducible** - Saved and loadable with joblib
4. ✅ **Production-ready** - Single function for inference